# Modelos con y sin cuotas: validación temporal

Cada rama compara seis familias de clasificación. Sin cuotas también se evalúa diferencia de goles convertida a probabilidades. Ningún resultado de test participa en esta etapa.

## Inicialización

Usamos la misma identidad de datos, código y protocolo en todos los cuadernos.

In [1]:
from pathlib import Path
import sys, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
ROOT = Path.cwd()
while not (ROOT/'scripts/mlcore').is_dir():
    if ROOT == ROOT.parent: raise RuntimeError('No se encontro la raiz del proyecto')
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT/'scripts'))
from mlcore.experiment import initialize, aligned
from mlcore.data import read_data, FOLDS, TEST
OUT = initialize()
print('Ejecucion verificada:', OUT.name)


Ejecucion verificada: 01afe756130bafc6


**Qué encontramos y cómo se usa.** La identidad evita mezclar artefactos de protocolos distintos. Los cálculos científicos viven en `scripts/mlcore`; estos cuadernos explican, ejecutan y muestran cada etapa.

## Ajustes interiores y predicciones exteriores

Las configuraciones de cada candidato se eligen con las dos temporadas anteriores a cada fold. Cada split se calcula una vez y se identifica por huellas de datos, código y protocolo.

In [2]:
from mlcore.experiment import nested_cv
if not (OUT/'oof_base.csv').exists(): nested_cv(OUT,jobs=4)
df, sets = read_data(OUT)
base = pd.read_csv(OUT/'oof_base.csv')
assert not base.Season.isin(TEST).any()
display(pd.read_csv(OUT/'model_comparison.csv'))

/home/evssz/UTEC/6to/machine/ML_Project/scripts/mlcore/data.py:290: DtypeWarning: Columns (0: Time, 1: B365CH) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(out/'dataset.csv', parse_dates=['Date', 'H_snapshot_at', 'A_snapshot_at'])


,candidate,n,log_loss,RPS,Brier,AUC_macro,accuracy,balanced_accuracy,ECE_masa,ECE_ancho,acierto_ganador_condicional,recall_empate
0,con/LR_REF,18184,0.965199,0.194988,0.572867,0.684391,0.538715,0.464498,0.008538,0.006877,0.716795,0.001992
1,con/weighted,18184,0.965218,0.194984,0.572874,0.684354,0.538440,0.464340,0.008883,0.006541,0.716502,0.001770
2,con/LR_consenso,18184,0.965737,0.195097,0.573167,0.683616,0.537945,0.463472,0.008736,0.007730,0.715990,0.000664
3,con/mean,18184,0.967337,0.195547,0.574265,0.681889,0.538165,0.463130,0.010523,0.009753,0.716209,0.000000
4,con/RF,18184,0.967576,0.195493,0.574173,0.681364,0.537615,0.464797,0.008830,0.007731,0.715185,0.005975
5,con/LR_MI5,18184,0.967937,0.195238,0.574332,0.679784,0.537670,0.463382,0.012590,0.008964,0.715477,0.000000
6,con/LR_MI10,18184,0.968173,0.195293,0.574439,0.679814,0.538165,0.463825,0.013031,0.009147,0.716136,0.000000
7,con/XGB,18184,0.968308,0.195689,0.574686,0.681327,0.537890,0.464026,0.012335,0.010611,0.715843,0.000664
8,con/HGB,18184,0.968530,0.195704,0.574816,0.680613,0.538495,0.464325,0.012702,0.009617,0.716795,0.000000
9,con/LR,18184,0.969894,0.196020,0.575624,0.679279,0.535691,0.464694,0.008755,0.007535,0.712258,0.020137


**Qué encontramos / decisión.** Las seis temporadas exteriores son comunes. Las predicciones de 2017/18 solo inicializan los ensembles; no se incluyen en el resultado principal.

## Identidad y alineación

Cada predicción se enlaza por identificador de partido, no por la posición de una tabla antigua.

In [3]:
for candidate,g in base.groupby('candidate'):
    assert not g.match_id.duplicated().any()
    check = g.merge(df[['match_id','y']],on='match_id',validate='one_to_one',suffixes=('_p','_d'))
    assert check.y_p.eq(check.y_d).all()
print('Predicciones alineadas:', base.candidate.nunique(), 'candidatos')

Predicciones alineadas: 29 candidatos


**Qué encontramos / decisión.** Estas predicciones son las que consumirán evaluación y ensembles; no hay un camino paralelo con un SVM anterior.